# Lesson 04 — Captions → Embeddings (The GPU Magic)

This is the core lesson. We run **CLIP's text encoder** on every BLIP-generated
caption from Lesson 03 to produce an **embedding** — a vector of 512 numbers
that captures the *meaning* of that caption.

### What is CLIP?
CLIP (Contrastive Language–Image Pretraining) learns to put images and text in the **same vector space**. The caption "a dog playing in a park" and a similar caption end up near each other. An unrelated caption is far away.

### Why GPU?
CLIP's text encoder is a transformer. On CPU it's still fast for short captions, but at scale the GPU processes 16 captions simultaneously, which is what makes this practical for large image collections.

## Step 1 — Load environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
S3_BUCKET = os.environ["S3_BUCKET"]
print(f"S3 bucket : {S3_BUCKET}")

## Step 2 — Submit the embedding job to Batch

The job will:
1. Download the caption manifest from `s3://<bucket>/captions/sample/manifest.json`
2. Embed each caption with CLIP's text encoder in batches of 16 (on the GPU)
3. Upsert one vector record per caption into the S3 Vectors index (`S3_VECTOR_BUCKET` / `S3_VECTOR_INDEX`)

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "submit_job.py", "--batch-stem", "sample"],
)
print("Exit code:", result.returncode)

## Step 3 — Query S3 Vectors to confirm the embeddings landed

In [ ]:
import boto3
import torch
from transformers import CLIPModel, CLIPProcessor

S3_VECTOR_BUCKET = os.environ["S3_VECTOR_BUCKET"]
S3_VECTOR_INDEX  = os.environ["S3_VECTOR_INDEX"]

s3vectors = boto3.client("s3vectors")

# Embed a throwaway query locally (CPU is fine for a single short string)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

with torch.no_grad():
    inputs   = processor(text=["a photo"], return_tensors="pt", padding=True)
    features = model.get_text_features(**inputs)
    features = features / features.norm(dim=-1, keepdim=True)

query_vector = features[0].numpy().astype("float32").tolist()

resp = s3vectors.query_vectors(
    vectorBucketName=S3_VECTOR_BUCKET,
    indexName=S3_VECTOR_INDEX,
    queryVector={"float32": query_vector},
    topK=5,
    returnMetadata=True,
)

print(f"S3 Vectors returned {len(resp['vectors'])} results:")
for match in resp["vectors"]:
    meta = match["metadata"]
    print(f"  {meta['image_key']}  →  {meta['caption']!r}  (distance: {match['distance']:.4f})")

## Step 4 — Visualise: project sampled embeddings to 2D with PCA

512 dimensions are hard to see. As an optional diagnostic, we sample a handful
of stored embeddings back out via `QueryVectors` and use PCA to squash them to
2D so we can plot the captions as dots. Captions with similar meaning cluster
together.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Optional diagnostic only — embed a small, explicitly chosen sample of captions
# locally (not the full stored index) just to see how CLIP spreads them out.
sample_captions = [
    "a dog playing in a park",
    "a busy city street",
    "a plate of food",
    "a mountain landscape",
    "a person riding a bicycle",
]

with torch.no_grad():
    inputs   = processor(text=sample_captions, return_tensors="pt", padding=True)
    features = model.get_text_features(**inputs)
    features = features / features.norm(dim=-1, keepdim=True)

sample_embeddings = features.numpy()

pca    = PCA(n_components=2)
coords = pca.fit_transform(sample_embeddings)   # shape: (N, 2)

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], s=60)
for (x, y), caption in zip(coords, sample_captions):
    plt.annotate(caption, (x, y), fontsize=8, wrap=True)
plt.title("Sample caption embeddings projected to 2D (PCA)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.tight_layout()
plt.show()

print(f"Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.1%}")

## Key Takeaway

> CLIP turns every caption into a 512-number fingerprint. Captions with similar meaning → similar fingerprints → close in space. **That's the magic we'll use in Lesson 05 to search by text.**

---

## Next lesson → [05 — Vector Search](../05-vector-search/notebook.ipynb)

We'll type a text query and find the matching image, via its caption — no SQL, no tags, just math on vectors.